
# **Notebook Pelatihan Model Analisis Sentimen - Ulasan **



## I. SETUP LINGKUNGAN DAN IMPOR LIBRARY


### 1.1. Instalasi Library yang Diperlukan

In [2]:
# 1. INSTALASI LIBRARY
!pip install pandas numpy scikit-learn tensorflow keras langdetect Sastrawi google-play-scraper nltk

import pandas as pd
import numpy as np
import re
import string
import os
import time
import requests # Untuk download Kamus Bahasa Alay dari GitHub

# Library Scrapping
from google_play_scraper import Sort, reviews, app

# Library NLP & Preprocessing
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory

# Download data pendukung NLTK untuk tokenisasi
nltk.download('punkt')

# Library Visualisasi (Penting untuk EDA awal)
import matplotlib.pyplot as plt
import seaborn as sns

# Library Deep Learning & ML (Tetap seperti sebelumnya)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

# Setup Reproduksibilitas
DetectorFactory.seed = 42

print("✅ Setup Library Selesai. Siap melakukan scrapping dan preprocessing.")

  Using cached langdetect-1.0.9.tar.gz (981 kB)
  Preparing metadata (setup.py) ... done
  Using cached Sastrawi-1.0.1-py2.py3-none-any.whl.metadata (909 bytes)
  Using cached google_play_scraper-1.2.7-py3-none-any.whl.metadata (50 kB)
Using cached Sastrawi-1.0.1-py2.py3-none-any.whl (209 kB)
Using cached google_play_scraper-1.2.7-py3-none-any.whl (28 kB)
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=a9d0816ca006750671b4ee9da308412e219197341671bd3310d3675e16e24340
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


✅ Setup Library Selesai. Siap melakukan scrapping dan preprocessing.


### 1.2. Load Referensi

In [3]:
# --- 1. KAMUS ALAY (Normalisasi) ---
url_alay = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
df_alay = pd.read_csv(url_alay)
alay_dict = dict(zip(df_alay['slang'], df_alay['formal']))

# --- 2. LEXICON INSET (Labeling) ---
# Kita ambil yang positif dan negatif
url_inset_pos = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
url_inset_neg = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

df_inset_pos = pd.read_csv(url_inset_pos, sep='\t')
df_inset_neg = pd.read_csv(url_inset_neg, sep='\t')
# Gabungkan jadi satu dictionary untuk cek skor kata
inset_dict = dict(zip(df_inset_pos['word'], df_inset_pos['weight']))
inset_dict.update(dict(zip(df_inset_neg['word'], df_inset_neg['weight'])))

# --- 3. COMBINED STOPWORDS (Pembersihan) ---
url_stopwords = "https://raw.githubusercontent.com/louisowen6/NLP_bahasa_resources/master/combined_stop_words.txt"
response = requests.get(url_stopwords)
# Simpan dalam bentuk set agar pencarian kata lebih cepat
combined_stopwords = set(response.text.splitlines())

print(f"✅ Kamus Alay: {len(alay_dict)} kata")
print(f"✅ InSet Lexicon: {len(inset_dict)} kata")
print(f"✅ Combined Stopwords: {len(combined_stopwords)} kata")

✅ Kamus Alay: 4331 kata
✅ InSet Lexicon: 9074 kata
✅ Combined Stopwords: 675 kata



## II. DATA ACQUISITION (SCRAPPING)



### 2.1. Scrapping ulasan M-Paspor dari Google Play

In [5]:
# PROSES SCRAPPING DATA M-PASPOR
from google_play_scraper import Sort, reviews

print("Memulai proses scrapping ulasan M-Paspor...")

result, continuation_token = reviews(
    'id.go.imigrasi.paspor_online',
    lang='id',      # Kita filter bahasa Indonesia saja
    country='id',   # Wilayah Indonesia
    sort=Sort.NEWEST,
    count=5000,     # Ambil 5.000 ulasan
    filter_score_with=None # Ambil semua rating (1-5)
)

# Ubah ke DataFrame
df_raw = pd.DataFrame(result)

# Pilih kolom yang kita butuhkan saja
df = df_raw[['content', 'score', 'at']].copy()
df.rename(columns={'content': 'text', 'score': 'rating', 'at': 'date'}, inplace=True)

print(f"✅ Berhasil mengambil {len(df)} ulasan.")
print(df.head())

# Simpan ke folder data/ (opsional kalau di Colab)
df.to_csv('raw_reviews_mpaspor.csv', index=False)

Memulai proses scrapping ulasan M-Paspor...
✅ Berhasil mengambil 5000 ulasan.
                                                text  rating  \
0  saran saya ga usah lewat aplikasi.. pasti ribe...       1   
1  aplikasi basi, tidak bisa dipakai, mungkin sen...       1   
2  baru juga tgl 1 maret jam 01:00 WIB slot bikin...       1   
3  pengurusan sampai saat ini masih baik² belum a...       5   
4  sudah dicoba, lumayan oke, hanya kendala setel...       2   

                 date  
0 2026-03-02 07:35:06  
1 2026-03-01 12:46:26  
2 2026-02-28 18:34:24  
3 2026-02-27 12:24:28  
4 2026-02-27 03:14:24  


In [6]:
all_reviews = []
continuation_token = None
target_count = 15000

print(f"⏳ Memulai scrapping target {target_count} ulasan...")

while len(all_reviews) < target_count:
    result, continuation_token = reviews(
        'id.go.imigrasi.paspor_online',
        lang='id',
        country='id',
        sort=Sort.NEWEST,
        count=5000, # Maksimal per tarikan
        continuation_token=continuation_token
    )
    all_reviews.extend(result)
    print(f"✅ Terambil: {len(all_reviews)} ulasan...")

    if not continuation_token: # Berhenti kalau sudah habis
        break

df_raw = pd.DataFrame(all_reviews)
df = df_raw[['content', 'score', 'at']].copy()
df.rename(columns={'content': 'text', 'score': 'rating', 'at': 'date'}, inplace=True)

print(f"Total Akhir: {len(df)} ulasan berhasil didownload!")

⏳ Memulai scrapping target 15000 ulasan...
✅ Terambil: 5000 ulasan...
✅ Terambil: 10000 ulasan...
✅ Terambil: 15000 ulasan...
🔥 Total Akhir: 15000 ulasan berhasil didownload!


### 2.2. Inisiasi Stemmer & Fungsi Preprocessing

In [9]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("Menyiapkan Stemmer Sastrawi...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()
print("✅ Stemmer siap!")

Menyiapkan Stemmer Sastrawi...
✅ Stemmer siap!


In [11]:
def preprocess_v1(text):
    # BASIC CLEANING
    text = str(text).lower() # Case folding
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'[-+]?[0-9]+', '', text)           # Hapus angka
    text = re.sub(r'[^\w\s]', '', text)               # Hapus tanda baca

    # NORMALISASI ALAY
    words = text.split()
    normalized_words = [alay_dict.get(word, word) for word in words]
    text = " ".join(normalized_words)

    # STEMMING (Mencari akar kata - Sastrawi)
    text = stemmer.stem(text)

    # TOKENISASI
    tokens = word_tokenize(text)

    # STOPWORDS REMOVAL
    tokens = [word for word in tokens if word not in combined_stopwords and len(word) > 1]

    return tokens

### 2.3. Eksekusi ke DataFrame

In [14]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Coba dulu 100
print("⏳ Memulai Preprocessing & Tokenisasi (Sampel 100 data)...")

# kolom baru 'tokens'
df['tokens'] = df['text'].head(100).apply(preprocess_v1)

print("✅ Selesai!")
print(df[['text', 'tokens']].head(10))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


⏳ Memulai Preprocessing & Tokenisasi (Sampel 100 data)...
✅ Selesai!
                                                text  \
0  saran saya ga usah lewat aplikasi.. pasti ribe...   
1  aplikasi basi, tidak bisa dipakai, mungkin sen...   
2  baru juga tgl 1 maret jam 01:00 WIB slot bikin...   
3  pengurusan sampai saat ini masih baik² belum a...   
4  sudah dicoba, lumayan oke, hanya kendala setel...   
5  gak niat banget buat aplikasinya. aplikasi gak...   
6  sudah menunggu 3 hari untuk menentukan tanggal...   
7  gak bisa masuk.. selalu minta aktivasi, tapi b...   
8  disarankan jangan dulu, lebih baik yg bisa bay...   
9          giliran pilih tanggal jadwal koq gak bisa   

                                              tokens  
0  [saran, aplikasi, ribet, bikin, ribet, beda, c...  
1  [aplikasi, basi, pakai, mungkin, sengaja, baha...  
2  [tanggal, maret, jam, slot, bikin, paspor, mad...  
3                              [urus, sulit, lapang]  
4  [coba, lumayan, oke, kendala, bayar,

###*Checkpoint Week 1*

In [15]:
df.to_csv('Tugas_1A_Mpaspor_Tokenized.csv', index=False)
print("✅ File tersimpan!")

✅ File tersimpan!


In [16]:
!pip freeze > requirements.txt

## III. PREPROCESSING TEKS

### 3.1.

## IV. PREPROCESSING TEKS BAHASA INDONESIA
### 4.1. Fungsi Pembersihan Teks Utama

## V. PERSIAPAN DATA UNTUK PELATIHAN MODEL
*Bagian ini dijalankan setelah Bagian IV selesai dan final_df sudah terbentuk*
### 5.1. Pemisahan Fitur dan Label

### 5.2. Encoding Label

### 5.3. Pembagian Data Pelatihan dan Pengujian
 *Kita perlu dua skema pembagian untuk 3 percobaan: 80/20 dan 70/30*

## VI. EKSTRAKSI FITUR
### 6.1. Tokenizer dan Padding untuk Deep Learning
*Mengubah teks menjadi urutan angka (indeks kata)*

### 6.2. TF-IDF untuk Machine Learning Klasik

## VII. IMPLEMENTASI TRAINING (3 skema)


## VIII. INFERENSI/PREDIKSI